In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_35_try_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 36 ###

def grab_subset_of_data_48(original_df, question_of_interest):
    # GPU‐accelerated column‐mask and selection
    col_mask = original_df.columns.str.contains(question_of_interest)
    df_subset = original_df.loc[:, col_mask]

    # pull the part after the dash with a purely GPU split/get/strip chain
    df_subset.columns = (
        df_subset.columns
            .str.split('-', n=1)
            .str.get(1)
            .str.strip()
    )

    # drop rows where *all* values are null using GPU boolean ops
    non_null_rows = df_subset.notnull().any(axis=1)
    df_subset = df_subset.loc[non_null_rows]
    return df_subset

question_of_interest_cell48 = (
    'Which of the following ML algorithms do you use on a regular basis?'
)
ml_algos_df_2022 = grab_subset_of_data_48(
    responses_df_2022_cell10,
    question_of_interest_cell48
)
ml_algos_df_2022.info()